In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.fft import fft, fftshift, fftfreq

# --- 1. Parâmetros e Geração do Sinal ---
fs = 40000                     # Frequência de amostragem
T_sym = 0.002                  # Período do símbolo
num_symbols = 500              # Número de símbolos
t = np.arange(0, num_symbols * T_sym, 1/fs)

# Gerando sinal m(t) com formato de pulso (Estilo Banda Larga/QAM)
np.random.seed(42)
symbols = np.random.choice([-3, -1, 1, 3], num_symbols)
m_t = np.zeros_like(t)

for i, sym in enumerate(symbols):
    t_center = i * T_sym
    # Pulso Sinc para criar uma largura de banda bem definida
    idx = (t > t_center - 10*T_sym) & (t < t_center + 10*T_sym)
    m_t[idx] += sym * np.sinc((t[idx] - t_center) / T_sym)

# --- 2. Modulação (AM-DSB-SC) ---
fc = 5000                      # Portadora de 5kHz
p_t = np.cos(2 * np.pi * fc * t)
r_t = m_t * p_t                # Sinal modulado (Transmitido/Recebido)

# --- 3. Demodulação e Filtragem (Fase Zero) ---
s_t = r_t * p_t                # Sinal misturado (Baseband + 2fc)

def filtro_passa_baixas(data, cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low')
    # filtfilt processa o sinal para frente e para trás para remover o atraso
    return filtfilt(b, a, data)

# Recuperação da mensagem com compensação de lag
# Podemos multiplicar por 2 para compensar a perda de modulação
m_recuperado = filtro_passa_baixas(s_t, 800, fs)*2

# --- 4. Função de Análise Espectral ---
def get_spectrum(signal, fs):
    n = len(signal)
    freqs = fftshift(fftfreq(n, 1/fs))
    # Espectro em dB para visualização clara das bandas secundárias
    spectrum = fftshift(np.abs(fft(signal)) / n)
    db_spectrum = 20 * np.log10(spectrum + 1e-6) 
    return freqs, db_spectrum

f, M_f = get_spectrum(m_t, fs)
_, R_f = get_spectrum(r_t, fs)
_, S_f = get_spectrum(s_t, fs)
_, M_rec_f = get_spectrum(m_recuperado, fs)

# --- 5. Plotagem dos Gráficos ---
plt.figure(figsize=(14, 16))

# Subplot 1: Modulação (Tempo)
plt.subplot(4, 2, 1)
plt.plot(t, m_t, 'b', label='m(t) Mensagem')
plt.plot(t, r_t, 'r', alpha=0.5, label='r(t) AM-DSB-SC')
plt.xlabel('Tempo (s)')
plt.ylabel('Amplitude (V)')
plt.xlim(0, 0.04)
plt.title('Domínio do Tempo: Modulação')
plt.legend()
plt.grid(True)

# Subplot 2: Modulação (Frequência)
plt.subplot(4, 2, 2)
plt.plot(f, M_f, 'b', label='M(f) Banda Base')
plt.plot(f, R_f, 'r', label='R(f) DSB-SC')
plt.title('Domínio da Frequência: Espectro Transmitido')
plt.ylabel('Magnitude (dB)')
plt.xlim(-10000, 10000)
plt.ylim(-70, 10)
plt.legend()
plt.grid(True)

# Subplot 3: Demodulação Bruta (Tempo)
plt.subplot(4, 2, 3)
plt.plot(t, s_t, 'orange', label='s(t) (Demodulado sem Filtro)')
plt.xlim(0, 0.04)
plt.title('Domínio do Tempo: Saída do Misturador (m(t) + 2fc)')
plt.ylabel('Amplitude (V)')
plt.legend()
plt.grid(True)

# Subplot 4: Demodulação Bruta (Frequência)
plt.subplot(4, 2, 4)
plt.plot(f, S_f, 'orange', label='S(f) Espectro')
plt.title('Domínio da Frequência: Banda Base e Componente em 2fc')
plt.xlim(-12000, 12000)
plt.ylabel('Magnitude (dB)')
plt.legend()
plt.grid(True)

# Subplot 5: Comparação Final (Tempo - Sem Atraso)
plt.subplot(4, 2, 5)
plt.plot(t, m_t, 'b', lw=2, label='Original m(t)')
plt.plot(t, m_recuperado, 'r--', lw=1.5, label='Recuperado m(t)')
plt.xlim(0, 0.04)
plt.title('Domínio do Tempo: Comparação m(t) Original vs Recuperado')
plt.xlabel('Tempo (s)')
plt.ylabel('Amplitude (V)')
plt.legend()
plt.grid(True)

# Subplot 6: Comparação Final (Frequência)
plt.subplot(4, 2, 6)
plt.plot(f, M_f, 'b', label='Original M(f)')
plt.plot(f, M_rec_f, 'r--', label='Recuperado M(f)')
plt.title('Domínio da Frequência: Comparação dos Espectros')
plt.xlim(-2000, 2000)
plt.ylabel('Magnitude (dB)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()